Dynawo Notebooks: PyPowSyBl Approach 1 for IEEE-57 Initialization

This notebook demonstrates **Approach 1** for the initialization of a large-scale Modelica 
power system model (specifically `IEEE57NoEvent.mo`). 

**Workflow:**
1. Parse the user's `.mo` files and extract the electrical topology.
2. Build an equivalent static PyPowSyBl network (`Network` object).
3. Compile and link dynamic preassembled models through PyPowSyBl (`ModelMapping`).
4. Delegate initialization to Dynawo via PyPowSyBl (Load Flow + INIT) using the `--dumpInit` flag.
5. Extract internal initialization parameters from the generated dump files.
6. Re-inject these parameters into the original Modelica case to leave it ready for OpenModelica users.

In [1]:
import os
import pandas as pd
import pypowsybl as pp
from IPython.display import display

# Internal framework imports
from dynawo_notebooks.Scripts.core.mo_topology import MoTopologyToolkit
from dynawo_notebooks.Scripts.core.model_linker import link_models
from dynawo_notebooks.Scripts.core.initialization_utils import (
    parse_all_dumps,
    reinject_into_modelica,
)

# Configuration constants for the IEEE-57 network
SOURCE_DIR = "../Models"
MODEL_NAME = "Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent"
DYNAWO_PKG_PATH = "/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"
LOCAL_FILES = ["IEEE57Base.mo", "IEEE57NoEvent.mo"]
MODELS_REGISTRY_PATH = "../Models/parsed_models_data.json"
OUTPUT_FILE = "IEEE57_initialized.mo"
EXPORT_FOLDER = f"grid_export_IEEE57"

print(f"PyPowSyBl version: {pp.__version__}")
print(f"Target Model: {MODEL_NAME}")

PyPowSyBl version: 1.15.0
Target Model: Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent



## Step 1: Parse Modelica File and Build Static PyPowSyBl Network
We use the `MoTopologyToolkit` facade to connect to OpenModelica, parse the 
abstract syntax tree (AST) of `IEEE57NoEvent.mo`, and force the topology into a strict 
PyPowSyBl IIDM static network.

In [2]:
# Initialize the toolkit to establish the ZMQ session with the OpenModelica Compiler (OMC)
toolkit = MoTopologyToolkit(
    source_dir=SOURCE_DIR,
    model_name=MODEL_NAME,
    dynawo_pkg_path=DYNAWO_PKG_PATH,
    local_files_list=LOCAL_FILES,
)

print("Parsing electrical data from the Modelica AST...")
# topology_data = toolkit.parse_electrical_data()

print("Translating parsed data into a PyPowSyBl static network...")

# Export for verification into the centralized folder
json_filepath = os.path.join(EXPORT_FOLDER, f"grid_export_IEEE57.json")
# toolkit.export_to_standard_json(topology_data, json_filepath)

# Import for verification
topology_data = toolkit.import_from_standard_json(json_filepath)

network = toolkit.build_powsybl_network(topology_data)

[OMC log for 'sendExpression(checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), True)']: [translation:warning:150] Connector switchOffSignal3 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpression(checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), True)']: [translation:warning:150] Connector QGenPu is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpression(checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), True)']: [translation:warning:150] Connector UPu is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpression(checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), True)']: [translation:warning:150] Connector QStatorPu is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC l

Parsing electrical data from the Modelica AST...
Translating parsed data into a PyPowSyBl static network...


## Step 2: Dynamic Model Linking (Recollement via PyPowSyBl)
Here we map the static PyPowSyBl elements to the specific Dynawo preassembled 
dynamic classes. We use the project's generic `link_models` function, which automatically 
iterates through all generators, loads, lines, and shunts across the entire grid.

In [3]:
print("Executing dynamic model linkage via the project's model linker...")

# The model_linker.py dynamically dispatches the correct API methods 
# based on the components found in the network.
model_mapping, mapping_summary_df = link_models(network, MODELS_REGISTRY_PATH)

if model_mapping is None:
    raise RuntimeError("CRITICAL ERROR: The mapping process failed. Check the JSON registry path.")

print("\n--- Dynamic Model Mapping Summary ---")
# Display the first 15 mapped elements to keep the notebook view clean for large networks
display(mapping_summary_df.head(15))

Executing dynamic model linkage via the project's model linker...

--- Dynamic Model Mapping Summary ---


,static_id,parameter_set_id,model_name
static_id,,,
Gen1,Gen1,Gen1,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Gen2,Gen2,Gen2,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Gen3,Gen3,Gen3,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Gen6,Gen6,Gen6,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Gen8,Gen8,Gen8,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Gen9,Gen9,Gen9,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Gen12,Gen12,Gen12,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
Load1,Load1,Load1,Dynawo.Electrical.Loads.LoadAlphaBeta
Load2,Load2,Load2,Dynawo.Electrical.Loads.LoadAlphaBeta


## Step 3: Initialization via Dynawo (PyPowSyBl Simulation)
We configure a dynamic simulation to stop at `t=0.0`. 
For this demonstration, we assume the simulation has already run and generated the 
`--dumpInit` files in the corresponding target directory, bypassing the Java execution 
to avoid current PyPowSyBl API limitations.

In [4]:
print("Preparing Dynawo dynamic simulation (Initialization phase)...")

# -------------------------------------------------------------------------
# DYNAWO EXECUTION
# Temporarily disabled pending native support for the --dumpInit flag
# within the PyPowSyBl Java core.
# -------------------------------------------------------------------------
# event_mapping = pp.dynamic.EventMapping()
# output_mapping = pp.dynamic.OutputVariableMapping()
# sim_parameters = pp.dynamic.Parameters(start_time=0.0, stop_time=0.0)
# simulation = pp.dynamic.Simulation()
# results = simulation.run(network, model_mapping, event_mapping=event_mapping, timeseries_mapping=output_mapping, parameters=sim_parameters)
# -------------------------------------------------------------------------

print("Bypassing execution: Utilizing pre-existing initialization dumps for the IEEE-57 network...")

dump_dir = "initValues_IEEE57/localInit"

if os.path.exists(dump_dir):
    print(f"Located initialization dump directory at: {dump_dir}")
else:
    print(f"WARNING: The directory {dump_dir} was not found. Please ensure the dump files are placed there.")

Preparing Dynawo dynamic simulation (Initialization phase)...
Bypassing execution: Utilizing pre-existing initialization dumps for the IEEE-57 network...
Located initialization dump directory at: initValues_IEEE57/localInit


## Step 4: Extract Parameters and Re-Inject into Modelica
We read the initialization dumps generated by Dynawo and inject these internal 
parameters back into the user's original `IEEE57NoEvent.mo` file using OpenModelica scripting.

In [5]:
# Parse the dumped directory utilizing the centralized utility functions
parsed_initialization_data = parse_all_dumps(dump_dir)

# Inject the extracted parameters back into the Modelica AST via OMC
reinject_into_modelica(toolkit.connector, MODEL_NAME, parsed_initialization_data, OUTPUT_FILE)

OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic parser result.
OMTypedParser error: Expected end of text. Returning the basic p